In [1]:
"""
export_flujos_detallados.py
-----------------------------
Complemento de playbook_automation.ipynb: exporta un CSV con el
detalle POR FLUJO INDIVIDUAL (no solo el conteo agregado que ya guarda
playbook_current.json).

Cada fila del CSV resultante responde directamente a la pregunta del
analista SOC: "de este lote evaluado, ¿cual flujo especifico es
Alto/Medio/Bajo/Benigno, y con que score?".

Reutiliza exactamente la misma logica de seleccion de modelo campeon,
scoring y severidad ya validada en playbook_automation.ipynb -- no
duplica ni reinventa esa logica, solo le agrega la salida detallada
que faltaba.

Notebook/script autocontenido: monta Drive, reconstruye el mismo test
split, recarga el modelo campeon desde su .pt, y vuelve a calcular
scores/severidad para poder exportarlos fila por fila.
"""

'\nexport_flujos_detallados.py\n-----------------------------\nComplemento de playbook_automation.ipynb: exporta un CSV con el\ndetalle POR FLUJO INDIVIDUAL (no solo el conteo agregado que ya guarda\nplaybook_current.json).\n\nCada fila del CSV resultante responde directamente a la pregunta del\nanalista SOC: "de este lote evaluado, ¿cual flujo especifico es\nAlto/Medio/Bajo/Benigno, y con que score?".\n\nReutiliza exactamente la misma logica de seleccion de modelo campeon,\nscoring y severidad ya validada en playbook_automation.ipynb -- no\nduplica ni reinventa esa logica, solo le agrega la salida detallada\nque faltaba.\n\nNotebook/script autocontenido: monta Drive, reconstruye el mismo test\nsplit, recarga el modelo campeon desde su .pt, y vuelve a calcular\nscores/severidad para poder exportarlos fila por fila.\n'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
PROJECT_DIR = "/content/drive/MyDrive/Trabajo_Cualitativo"
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

!jupyter nbconvert --to python "$PROJECT_DIR/preprocessing.ipynb"
!jupyter nbconvert --to python "$PROJECT_DIR/models.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Trabajo_Cualitativo/preprocessing.ipynb to python
[NbConvertApp] Writing 5391 bytes to /content/drive/MyDrive/Trabajo_Cualitativo/preprocessing.py
[NbConvertApp] Converting notebook /content/drive/MyDrive/Trabajo_Cualitativo/models.ipynb to python
[NbConvertApp] Writing 3362 bytes to /content/drive/MyDrive/Trabajo_Cualitativo/models.py


In [4]:
import os
import json

import numpy as np
import pandas as pd
import torch

from preprocessing import clean, encode_target, prepare_dataset
from models import build_model

# --------------------------------------------------------------------- #
# Configuracion (mismas rutas/semillas que train_pipeline.py y
# playbook_automation.ipynb)
# --------------------------------------------------------------------- #
PROJECT_DIR = "/content/drive/MyDrive/Trabajo_Cualitativo"
DATA_PATH = f"{PROJECT_DIR}/Data/02-15-2018.csv"
RESULTS_DIR = f"{PROJECT_DIR}/results"
SUMMARY_JSON_PATH = f"{RESULTS_DIR}/summary_statistics.json"
ALL_RUNS_PATH = f"{RESULTS_DIR}/all_runs.csv"
PLAYBOOK_DIR = f"{RESULTS_DIR}/playbook"
os.makedirs(PLAYBOOK_DIR, exist_ok=True)

SPLIT_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PRIMARY_METRIC = "f1_mean"
STD_TIEBREAKER = "f1_std"
SECONDARY_METRIC = "roc_auc_mean"
SEVERITY_PERCENTILES = (0.33, 0.66)

# Columnas "de contexto" que se reincorporan SOLO para identificar el
# flujo en el reporte (no se usan como feature del modelo, ya que
# Dst Port se excluye del entrenamiento por ser columna de leakage --
# ver preprocessing.py). Ayudan al analista a saber que investigar.
CONTEXT_COLS = ["Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts"]


def to_tensor(x, dtype=torch.float32):
    return torch.tensor(x, dtype=dtype, device=DEVICE)


# --------------------------------------------------------------------- #
# 1) Modelo/corrida campeon (misma logica que playbook_automation.ipynb)
# --------------------------------------------------------------------- #
def select_champion():
    with open(SUMMARY_JSON_PATH, "r", encoding="utf-8") as f:
        summary_df = pd.DataFrame(json.load(f))
    all_runs_df = pd.read_csv(ALL_RUNS_PATH)

    ranked = summary_df.sort_values(
        by=[PRIMARY_METRIC, STD_TIEBREAKER, SECONDARY_METRIC],
        ascending=[False, True, False],
    ).reset_index(drop=True)
    champion_model = ranked.loc[0, "Modelo"]

    model_runs = all_runs_df[all_runs_df["model"] == champion_model]
    champion_row = model_runs.sort_values(by="f1", ascending=False).iloc[0]
    return champion_model, champion_row


# --------------------------------------------------------------------- #
# 2) Reconstruye el mismo test split, conservando columnas de contexto
#    (igual patron que bias_audit.py, pero aqui para reporte, no para
#    auditoria de sesgo)
# --------------------------------------------------------------------- #
def rebuild_test_split_with_context(data):
    raw = pd.read_csv(DATA_PATH)
    raw["Label"] = raw["Label"].astype(str).str.strip()
    raw = raw.replace([np.inf, -np.inf], np.nan)

    raw_clean, _ = clean(raw, drop_leakage=False)  # conserva Dst Port/Protocol
    raw_clean = raw_clean.dropna()
    raw_encoded = encode_target(raw_clean)

    context_available = [c for c in CONTEXT_COLS if c in raw_clean.columns]
    context_full = raw_clean[context_available].reset_index(drop=True)

    y_full = raw_encoded["y"].values
    idx_full = np.arange(len(y_full))

    from sklearn.model_selection import train_test_split
    idx_train, idx_temp, _, y_temp = train_test_split(
        idx_full, y_full, test_size=0.40, stratify=y_full, random_state=SPLIT_SEED
    )
    idx_val, idx_test, _, _ = train_test_split(
        idx_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SPLIT_SEED
    )

    assert len(idx_test) == len(data["y_test"]), (
        "El tamano del split reconstruido no coincide con el de "
        "preprocessing.prepare_dataset(); revisa que la limpieza sea identica."
    )

    return context_full.iloc[idx_test].reset_index(drop=True)


# --------------------------------------------------------------------- #
# 3) Carga modelo campeon y genera scores (misma logica que
#    playbook_automation.ipynb)
# --------------------------------------------------------------------- #
def load_champion_and_score(champion_model, champion_row, n_features, X):
    run_name = f"{champion_model}_seed{int(champion_row['seed'])}"
    artifact_path = f"{RESULTS_DIR}/model_{run_name}.pt"

    model = build_model(champion_model, n_features).to(DEVICE)
    model.load_state_dict(torch.load(artifact_path, map_location=DEVICE))
    model.eval()

    with torch.no_grad():
        if champion_model == "Autoencoder":
            X_t = to_tensor(X)
            recon = model(X_t)
            err = torch.mean((recon - X_t) ** 2, dim=1).cpu().numpy()
            score_dominant = -err
            dominant_label = int(champion_row["dominant_label_trained_on"])
            scores = score_dominant if dominant_label == 1 else -score_dominant
        else:
            logits = model(to_tensor(X))
            scores = torch.sigmoid(logits).cpu().numpy()

    return scores, run_name


def assign_risk_tiers(scores, y_pred):
    flagged_scores = scores[y_pred == 1]
    tiers = np.full(len(scores), "Benigno", dtype=object)
    if len(flagged_scores) == 0:
        return tiers
    q_low, q_high = np.quantile(flagged_scores, list(SEVERITY_PERCENTILES))
    for i in range(len(scores)):
        if y_pred[i] == 0:
            continue
        if scores[i] >= q_high:
            tiers[i] = "Alto"
        elif scores[i] >= q_low:
            tiers[i] = "Medio"
        else:
            tiers[i] = "Bajo"
    return tiers


# --------------------------------------------------------------------- #
# 4) Orquestacion: arma el CSV fila por fila
# --------------------------------------------------------------------- #
def export_flujos_detallados():
    champion_model, champion_row = select_champion()
    print(f"Modelo campeon: {champion_model} (seed={int(champion_row['seed'])})")

    data = prepare_dataset(DATA_PATH, random_state=SPLIT_SEED)
    context_test = rebuild_test_split_with_context(data)

    scores, run_name = load_champion_and_score(
        champion_model, champion_row, data["n_features"], data["X_test"]
    )
    threshold = float(champion_row["threshold"])
    y_pred = (scores >= threshold).astype(int)
    y_true = data["y_test"]
    tiers = assign_risk_tiers(scores, y_pred)

    report = context_test.copy()
    report.insert(0, "flujo_id", range(len(report)))
    report["score_modelo"] = scores
    report["es_ataque_predicho"] = y_pred
    report["es_ataque_real"] = y_true
    report["nivel_severidad"] = tiers
    report["prediccion_correcta"] = (report["es_ataque_predicho"] == report["es_ataque_real"])

    # Orden util para el analista: primero lo mas critico
    orden_severidad = {"Alto": 0, "Medio": 1, "Bajo": 2, "Benigno": 3}
    report["_orden"] = report["nivel_severidad"].map(orden_severidad)
    report = report.sort_values(by=["_orden", "score_modelo"], ascending=[True, False]).drop(columns=["_orden"])

    out_path = f"{PLAYBOOK_DIR}/flujos_detallados_{run_name}.csv"
    report.to_csv(out_path, index=False)

    print(f"\nTotal de flujos exportados: {len(report)}")
    print(report["nivel_severidad"].value_counts())
    print(f"\nArchivo generado: {out_path}")
    print("\nPrimeras filas (las mas criticas):")
    print(report.head(10).to_string(index=False))

    return report, out_path




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
report, out_path = export_flujos_detallados()

Modelo campeon: MLP (seed=43)

Total de flujos exportados: 208110
nivel_severidad
Benigno    197650
Alto         3752
Bajo         3447
Medio        3261
Name: count, dtype: int64

Archivo generado: /content/drive/MyDrive/Trabajo_Cualitativo/results/playbook/flujos_detallados_MLP_seed43.csv

Primeras filas (las mas criticas):
 flujo_id  Dst Port  Protocol  Flow Duration  Tot Fwd Pkts  Tot Bwd Pkts  score_modelo  es_ataque_predicho  es_ataque_real nivel_severidad  prediccion_correcta
      158        80         6       36809105             2             0           1.0                   1               1            Alto                 True
      186        80         6      108038681            15             3           1.0                   1               1            Alto                 True
      207        80         6      102958118             2             1           1.0                   1               1            Alto                 True
      230        80         6   